# PromptPotter Optimizer - Advanced Usage

This notebook demonstrates advanced features:
- Custom instructions
- Different target metrics
- Batch optimization
- Result comparison

In [ ]:
!pip install requests pandas matplotlib -q

In [ ]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint

API_URL = "http://localhost:8000/api/v1"

## Advanced Example 1: Custom Instructions

In [ ]:
# Dataset for entity extraction
dataset = [
    {
        "text": "John Smith works at Microsoft in Seattle.",
        "expected": {"person": "John Smith", "company": "Microsoft", "location": "Seattle"}
    },
    {
        "text": "Apple CEO Tim Cook announced new products.",
        "expected": {"person": "Tim Cook", "company": "Apple"}
    },
]

# Custom instructions for optimization
custom_instructions = """
Focus on:
1. Extracting entities in consistent JSON format
2. Handling missing entities gracefully
3. Being concise and accurate
"""

response = requests.post(
    f"{API_URL}/optimize",
    json={
        "initial_prompt": "Extract entities from the text:",
        "dataset": dataset,
        "target_metric": "accuracy",
        "custom_instructions": custom_instructions,
        "max_iterations": 3
    }
)

result = response.json()
print("Optimized Prompt with Custom Instructions:")
print(result['optimized_prompt'])

## Advanced Example 2: Comparing Different Initial Prompts

In [ ]:
# Common dataset
dataset = [
    {"text": "Great product!", "expected": "positive"},
    {"text": "Awful experience", "expected": "negative"},
    {"text": "Love it!", "expected": "positive"},
    {"text": "Not good", "expected": "negative"},
]

# Different initial prompts to compare
initial_prompts = [
    "Classify sentiment:",
    "What is the sentiment of this text?",
    "Analyze the emotional tone:",
    "Is this positive or negative?"
]

results = []

for prompt in initial_prompts:
    response = requests.post(
        f"{API_URL}/optimize",
        json={
            "initial_prompt": prompt,
            "dataset": dataset,
            "max_iterations": 3
        }
    )
    
    if response.status_code == 200:
        result = response.json()
        results.append({
            "initial": prompt,
            "initial_score": result['initial_score'],
            "final_score": result['final_score'],
            "improvement": result['improvement']
        })

# Display comparison
df = pd.DataFrame(results)
print("\nComparison of Initial Prompts:")
print(df.to_string(index=False))

## Advanced Example 3: Visualizing Optimization Progress

In [ ]:
# Run optimization with more iterations
response = requests.post(
    f"{API_URL}/optimize",
    json={
        "initial_prompt": "Classify:",
        "dataset": dataset,
        "max_iterations": 8
    }
)

result = response.json()

# Extract scores from each iteration
iterations = [i['iteration'] for i in result['iterations']]
scores = [i['score'] for i in result['iterations']]

# Plot progress
plt.figure(figsize=(10, 6))
plt.plot(iterations, scores, marker='o', linewidth=2, markersize=8)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Optimization Progress Over Iterations', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal Improvement: {result['improvement']:.2f}%")

## Advanced Example 4: Loading Dataset from CSV

In [ ]:
# Example: Load dataset from CSV file
# Assuming CSV has 'text' and 'label' columns

# df = pd.read_csv('your_dataset.csv')
# dataset = df[['text', 'label']].rename(columns={'label': 'expected'}).to_dict('records')

# For demo, create sample dataset
df = pd.DataFrame({
    'text': [
        'Excellent service!',
        'Poor quality',
        'Highly recommended',
        'Waste of money'
    ],
    'label': ['positive', 'negative', 'positive', 'negative']
})

dataset = df.rename(columns={'label': 'expected'}).to_dict('records')

response = requests.post(
    f"{API_URL}/optimize",
    json={
        "initial_prompt": "Classify the sentiment:",
        "dataset": dataset,
        "max_iterations": 5
    }
)

result = response.json()
print(f"Optimized for {len(dataset)} examples")
print(f"Final score: {result['final_score']:.2%}")

## Helper Function: Reusable Optimization Client

In [ ]:
class PromptPotterClient:
    def __init__(self, api_url):
        self.api_url = api_url
    
    def optimize(self, prompt, dataset, **kwargs):
        """Optimize a prompt with simplified interface"""
        response = requests.post(
            f"{self.api_url}/optimize",
            json={
                "initial_prompt": prompt,
                "dataset": dataset,
                **kwargs
            }
        )
        response.raise_for_status()
        return response.json()
    
    def health(self):
        """Check API health"""
        response = requests.get(f"{self.api_url}/health")
        return response.json()

# Usage
client = PromptPotterClient(API_URL)

# Check health
print("API Health:", client.health())

# Optimize
result = client.optimize(
    prompt="Classify:",
    dataset=dataset,
    max_iterations=5
)

print(f"\nOptimized Prompt: {result['optimized_prompt']}")
print(f"Improvement: {result['improvement']:.2f}%")

## Next Steps

- Experiment with your own datasets
- Try different LLM models
- Implement custom metrics
- Build automated prompt testing pipelines